# 14d — SHAP on the Q-Val Transformer

Notebooks 14a/b/c ran TreeSHAP on a GradientBoosting surrogate trained on 10 hand-level features.  
This notebook runs GradientExplainer directly on the **actual trained transformer** (3.3M params, 6L/8H/256d) to get token-level attribution at the value head.

**Question**: Does the transformer confirm the n_doubles + trump_count story, or does it key on something else?

**SHAP variant used**: `shap.GradientExplainer` applied to the *embedding* layer outputs (continuous float tensors).  
The embedding vectors are computed once, frozen, and fed as input to a `ValueHeadWrapper` module  
that runs input_proj → TransformerEncoder → value_head.  
This gives gradients through the transformer without needing integer-input SHAP.

In [1]:
# === CONFIGURATION ===
PROJECT_ROOT = "/Users/jason/code/mk5-main"
CKPT_PATH = f"{PROJECT_ROOT}/forge/models/domino-qval-3.3M-shuffle-qgap0.074-qmae0.96.ckpt"
FIGURES_DIR = f"{PROJECT_ROOT}/forge/analysis/results/figures"
TABLES_DIR  = f"{PROJECT_ROOT}/forge/analysis/results/tables"
SHAP_14A_CSV = f"{TABLES_DIR}/14a_shap_importance.csv"

N_STATES   = 200   # root states to explain (precedent from 14a)
N_BG       = 100   # background set for GradientExplainer
DECL_ID    = 3     # sixes trump (non-trivial, has counts and trump structure)
CURRENT_PLAYER = 0 # P0's perspective
RANDOM_SEED    = 42

import sys
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch
import shap
from pathlib import Path

from forge.oracle.rng import deal_from_seed
from forge.oracle.tables import DOMINO_HIGH, DOMINO_LOW, DOMINO_IS_DOUBLE, DOMINO_COUNT_POINTS
from forge.ml.tokenize import (
    TRUMP_RANK_TABLE, COUNT_VALUE_MAP,
    TOKEN_TYPE_CONTEXT, TOKEN_TYPE_PLAYER0,
    MAX_TOKENS, N_FEATURES,
)
from forge.ml.transformer import DominoTransformer
from forge.analysis.utils import viz

viz.setup_notebook_style()
Path(FIGURES_DIR).mkdir(parents=True, exist_ok=True)
Path(TABLES_DIR).mkdir(parents=True, exist_ok=True)

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

print(f"shap {shap.__version__}, torch {torch.__version__}")
print("Ready")

shap 0.51.0, torch 2.11.0
Ready


## 1. Load the transformer

In [2]:
# The checkpoint was saved with torch.compile, so weights have _orig_mod prefix.
# Strip that prefix and load directly into DominoTransformer.
ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
hp = ckpt["hyper_parameters"]
print("Hyperparameters:", {k: v for k, v in hp.items() if k in ("embed_dim", "n_heads", "n_layers", "ff_dim", "loss_mode")})

model = DominoTransformer(
    embed_dim=hp["embed_dim"],
    n_heads=hp["n_heads"],
    n_layers=hp["n_layers"],
    ff_dim=hp["ff_dim"],
    dropout=0.0,  # eval mode anyway
)
raw_sd = {
    k.replace("model._orig_mod.", "").replace("model.", ""): v
    for k, v in ckpt["state_dict"].items()
    if k.startswith("model.")
}
model.load_state_dict(raw_sd, strict=True)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded: {n_params/1e6:.2f}M parameters")

Hyperparameters: {'embed_dim': 256, 'n_heads': 8, 'n_layers': 6, 'ff_dim': 512, 'loss_mode': 'qvalue'}
Model loaded: 3.26M parameters


## 2. Build synthetic root states

No shard data is available on this machine — synthetic root states from `deal_from_seed` are used.  
Root state = depth 28 (all dominoes in hand), no trick in progress.

In [3]:
def make_root_token(seed, decl_id=DECL_ID, current_player=CURRENT_PLAYER):
    """Tokenize a deal at root state (depth=28, no trick). Returns (tokens, mask, hands)."""
    hands = deal_from_seed(seed)
    tokens = np.zeros((MAX_TOKENS, N_FEATURES), dtype=np.int8)
    mask   = np.zeros(MAX_TOKENS, dtype=np.int8)

    # Context token
    tokens[0, 9]  = TOKEN_TYPE_CONTEXT
    tokens[0, 10] = decl_id
    tokens[0, 11] = 0  # leader == current_player at root
    mask[0] = 1

    for p in range(4):
        normalized_p = (p - current_player + 4) % 4
        for li in range(7):
            gid = hands[p][li]
            fi  = p * 7 + li
            ti  = 1 + fi
            tokens[ti, 0] = DOMINO_HIGH[gid]
            tokens[ti, 1] = DOMINO_LOW[gid]
            tokens[ti, 2] = 1 if DOMINO_IS_DOUBLE[gid] else 0
            tokens[ti, 3] = COUNT_VALUE_MAP[DOMINO_COUNT_POINTS[gid]]
            tokens[ti, 4] = TRUMP_RANK_TABLE[(gid, decl_id)]
            tokens[ti, 5] = normalized_p
            tokens[ti, 6] = 1 if normalized_p == 0 else 0  # is_current
            tokens[ti, 7] = 1 if normalized_p == 2 else 0  # is_partner
            tokens[ti, 8] = 1  # all dominoes remaining
            tokens[ti, 9] = TOKEN_TYPE_PLAYER0 + p
            tokens[ti, 10] = decl_id
            tokens[ti, 11] = 0
            mask[ti] = 1
    # No trick tokens at root
    return tokens, mask, hands


# Build N_STATES root states
all_data = [make_root_token(s) for s in range(N_STATES)]
all_tokens = [d[0] for d in all_data]
all_masks  = [d[1] for d in all_data]
all_hands  = [d[2] for d in all_data]

# At root, every state has the same valid-token mask pattern (tokens 0-28 valid)
FIXED_MASK = torch.tensor(all_masks[0:1], dtype=torch.long)  # (1, 32)

print(f"Built {N_STATES} root states. Mask sum per state: {all_masks[0].sum()} (expect 29)")

Built 200 root states. Mask sum per state: 29 (expect 29)


## 3. Compute raw embeddings and run the model

In [4]:
def get_concat_embeds(tokens_list):
    """Concatenate all 12 embedding outputs to get the pre-input_proj tensor.
    Returns float tensor (N, 32, embed_raw_dim).
    """
    toks = torch.tensor(np.stack(tokens_list), dtype=torch.long)
    with torch.no_grad():
        e = torch.cat([
            model.high_pip_embed(toks[:, :, 0]),
            model.low_pip_embed(toks[:, :, 1]),
            model.is_double_embed(toks[:, :, 2]),
            model.count_value_embed(toks[:, :, 3]),
            model.trump_rank_embed(toks[:, :, 4]),
            model.player_id_embed(toks[:, :, 5]),
            model.is_current_embed(toks[:, :, 6]),
            model.is_partner_embed(toks[:, :, 7]),
            model.is_remaining_embed(toks[:, :, 8]),
            model.token_type_embed(toks[:, :, 9]),
            model.decl_embed(toks[:, :, 10]),
            model.leader_embed(toks[:, :, 11]),
        ], dim=-1)
    return e  # (N, 32, embed_raw_dim)


embeds_all = get_concat_embeds(all_tokens)  # (200, 32, 357)
print(f"Embedding tensor: {embeds_all.shape}  (N=200, seq=32, embed_raw_dim={embeds_all.shape[-1]})")

# Get model predictions (value head) for all states
tokens_t = torch.tensor(np.stack(all_tokens), dtype=torch.long)
masks_t  = torch.tensor(np.stack(all_masks),  dtype=torch.long)
player_t = torch.full((N_STATES,), CURRENT_PLAYER, dtype=torch.long)

with torch.no_grad():
    _, values = model(tokens_t, masks_t, player_t)

values_np = values.numpy()
print(f"Value head predictions: mean={values_np.mean():.3f}, std={values_np.std():.3f}, "
      f"range=[{values_np.min():.3f}, {values_np.max():.3f}]")

Embedding tensor: torch.Size([200, 32, 357])  (N=200, seq=32, embed_raw_dim=357)
Value head predictions: mean=0.206, std=0.567, range=[-0.990, 0.986]


## 4. SHAP via GradientExplainer on the embedding space

The `ValueHeadWrapper` takes the concatenated raw embeddings (continuous floats)  
and runs `input_proj → TransformerEncoder → value_head`.  
GradientExplainer computes expected gradients with respect to a background distribution.

In [5]:
class ValueHeadWrapper(torch.nn.Module):
    """Wraps DominoTransformer to accept pre-embedding tensors.
    
    Input:  x_embed (B, 32, embed_raw_dim) — concatenated sub-embedding outputs
    Output: value   (B, 1)               — value head prediction
    """
    def __init__(self, inner_model, fixed_mask):
        super().__init__()
        self.m = inner_model
        self.register_buffer("fixed_mask", fixed_mask)  # (1, 32)

    def forward(self, x_embed):
        B = x_embed.shape[0]
        x = self.m.input_proj(x_embed)
        attn_mask = (self.fixed_mask.expand(B, -1) == 0)
        x = self.m.transformer(x, src_key_padding_mask=attn_mask)
        value = self.m.value_head(x[:, 0, :]).squeeze(-1)
        return value.unsqueeze(1)


wrapper = ValueHeadWrapper(model, FIXED_MASK)
wrapper.eval()

# Verify wrapper output matches full model
with torch.no_grad():
    wrapper_vals = wrapper(embeds_all).squeeze(1).numpy()
print(f"Max delta wrapper vs full model: {np.abs(wrapper_vals - values_np).max():.2e}  (should be ~0)")

Max delta wrapper vs full model: 0.00e+00  (should be ~0)


In [6]:
# GradientExplainer: background = first N_BG states, test = remaining 100
bg_embeds   = embeds_all[:N_BG]     # (100, 32, 357)
test_embeds = embeds_all[N_BG:]     # (100, 32, 357)
test_values = values_np[N_BG:]      # (100,)

print(f"Background set: {bg_embeds.shape[0]} states")
print(f"Test set:       {test_embeds.shape[0]} states")
print("Computing GradientExplainer SHAP values (may take ~30-60s)...")

explainer = shap.GradientExplainer(wrapper, bg_embeds)
# shap_values shape: (N_test, 32, embed_raw_dim, 1) — last dim is output dim
shap_vals_raw = explainer.shap_values(test_embeds)
shap_vals = np.array(shap_vals_raw).squeeze(-1)  # (100, 32, 357)

print(f"SHAP values shape: {shap_vals.shape}  (N_test, seq_len=32, embed_raw_dim=357)")
print(f"SHAP sum check (should ≈ pred - E[bg pred]):")
shap_sum_per_sample = shap_vals.sum(axis=(1, 2))  # (100,)
bg_mean_pred = wrapper(bg_embeds).detach().numpy().mean()
expected_delta = test_values - bg_mean_pred
print(f"  mean |SHAP_sum - (pred - E_bg)|: {np.abs(shap_sum_per_sample - expected_delta).mean():.4f}")
print("Done.")

Background set: 100 states
Test set:       100 states
Computing GradientExplainer SHAP values (may take ~30-60s)...


SHAP values shape: (100, 32, 357)  (N_test, seq_len=32, embed_raw_dim=357)
SHAP sum check (should ≈ pred - E[bg pred]):
  mean |SHAP_sum - (pred - E_bg)|: 0.1095
Done.


## 5. Per-token importance

Aggregate |SHAP| over the embedding dimension to get one importance score per token position.

In [7]:
# token_importance[i, j] = sum of |SHAP| over embedding dims for state i, token j
token_importance = np.abs(shap_vals).sum(axis=-1)   # (100, 32)
mean_token_imp   = token_importance.mean(axis=0)    # (32,)

# Token layout at root state:
#   0:      context token
#   1-7:    P0 hand (current player)
#   8-14:   P1 hand (left opponent)
#   15-21:  P2 hand (partner)
#   22-28:  P3 hand (right opponent)
#   29-31:  trick tokens (unused at root — zero importance expected)

TOKEN_LABELS = ["ctx"]
for p in range(4):
    for li in range(7):
        TOKEN_LABELS.append(f"P{p}_{li}")
TOKEN_LABELS += ["trick0", "trick1", "trick2"]

token_df = pd.DataFrame({
    "token_idx":       list(range(32)),
    "token_label":     TOKEN_LABELS,
    "mean_abs_shap":   mean_token_imp,
    "player":          [-1] + [p for p in range(4) for _ in range(7)] + [-1, -1, -1],
})

print("Top 15 tokens by mean |SHAP|:")
print(token_df.nlargest(15, "mean_abs_shap")[["token_label", "mean_abs_shap", "player"]].to_string(index=False))

Top 15 tokens by mean |SHAP|:
token_label  mean_abs_shap  player
       P3_6       0.238673       3
       P2_6       0.206404       2
       P3_4       0.199654       3
       P3_5       0.195518       3
       P3_3       0.191515       3
       P0_6       0.189963       0
       P1_6       0.183837       1
       P3_2       0.179280       3
       P2_3       0.175832       2
       P0_5       0.175013       0
       P2_2       0.171107       2
       P1_4       0.169736       1
       P0_3       0.168300       0
       P1_3       0.167648       1
       P1_5       0.165853       1


In [8]:
# Plot: per-token mean |SHAP| bar chart
fig, ax = plt.subplots(figsize=(14, 4))

colors = {
    -1: "#999999",   # context / trick
     0: "#2196F3",   # P0 (current)
     1: "#FF5722",   # P1 (opponent)
     2: "#4CAF50",   # P2 (partner)
     3: "#FF5722",   # P3 (opponent)
}
bar_colors = [colors[p] for p in token_df["player"]]

ax.bar(token_df["token_idx"], token_df["mean_abs_shap"], color=bar_colors, edgecolor="none")
ax.set_xticks(token_df["token_idx"])
ax.set_xticklabels(TOKEN_LABELS, rotation=90, fontsize=7)
ax.set_xlabel("Token position")
ax.set_ylabel("Mean |SHAP| (embedding space)")
ax.set_title("Per-token importance at value head (GradientExplainer, n=100 test states)")

legend_patches = [
    mpatches.Patch(color="#2196F3", label="P0 (current)"),
    mpatches.Patch(color="#4CAF50", label="P2 (partner)"),
    mpatches.Patch(color="#FF5722", label="P1/P3 (opponents)"),
    mpatches.Patch(color="#999999", label="Context/Trick"),
]
ax.legend(handles=legend_patches, loc="upper right", fontsize=8)
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/14d_shap_token_importance.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved 14d_shap_token_importance.png")

Saved 14d_shap_token_importance.png


## 6. Aggregate to per-domino importance

Each token in the hand encodes one domino.  
We look up which domino is in each token position across the 100 test states and compute mean |SHAP|.

In [9]:
# For each test state, map token positions 1-28 to the global domino ID they encode
# Token 1+flat_idx encodes hands[flat_idx // 7][flat_idx % 7]
test_hands = all_hands[N_BG:]  # 100 test hands

# Build a records list: (domino_id, mean_abs_shap for that token)
records = []
for i, (hands, state_imp) in enumerate(zip(test_hands, token_importance)):
    for p in range(4):
        for li in range(7):
            flat_idx = p * 7 + li
            ti = 1 + flat_idx
            gid = hands[p][li]
            records.append({
                "state_idx":    i,
                "token_idx":    ti,
                "player":       p,
                "domino_id":    gid,
                "high_pip":     DOMINO_HIGH[gid],
                "low_pip":      DOMINO_LOW[gid],
                "is_double":    bool(DOMINO_IS_DOUBLE[gid]),
                "count_pts":    DOMINO_COUNT_POINTS[gid],
                "token_shap":   state_imp[ti],
            })

rec_df = pd.DataFrame(records)

# Per-domino mean |SHAP|, aggregated across all states where that domino appears
per_domino = (
    rec_df.groupby(["domino_id", "high_pip", "low_pip", "is_double", "count_pts"])
    .agg(
        mean_abs_shap=("token_shap", "mean"),
        n_appearances=("token_shap", "count"),
    )
    .reset_index()
    .sort_values("mean_abs_shap", ascending=False)
)

# Label: "H-L" (e.g. "6-6")
per_domino["label"] = per_domino.apply(
    lambda r: f"{int(r.high_pip)}-{int(r.low_pip)}", axis=1
)

print("Top 10 dominoes by mean |SHAP| at value head:")
print(per_domino[["label", "mean_abs_shap", "is_double", "count_pts", "n_appearances"]].head(10).to_string(index=False))

# Save to table
per_domino.to_csv(f"{TABLES_DIR}/14d_shap_per_domino.csv", index=False)
token_df.to_csv(f"{TABLES_DIR}/14d_shap_token_importance.csv", index=False)
print("Saved 14d_shap_per_domino.csv and 14d_shap_token_importance.csv")

Top 10 dominoes by mean |SHAP| at value head:
label  mean_abs_shap  is_double  count_pts  n_appearances
  6-3       0.339045      False          0            100
  3-3       0.313469       True          0            100
  5-5       0.244401       True         10            100
  5-3       0.230166      False          0            100
  6-4       0.206939      False         10            100
  3-2       0.204743      False          5            100
  4-3       0.202501      False          0            100
  6-6       0.181911       True          0            100
  5-0       0.173550      False          5            100
  4-1       0.167560      False          5            100
Saved 14d_shap_per_domino.csv and 14d_shap_token_importance.csv


In [10]:
# Plot per-domino importance
top28 = per_domino.head(28).copy()  # all 28 dominoes

bar_c = ["#E91E63" if r.is_double else
          ("#FF9800" if r.count_pts > 0 else "#607D8B")
          for _, r in top28.iterrows()]

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(range(len(top28)), top28["mean_abs_shap"], color=bar_c)
ax.set_xticks(range(len(top28)))
ax.set_xticklabels(top28["label"], rotation=90, fontsize=8)
ax.set_xlabel("Domino (sorted by mean |SHAP|)")
ax.set_ylabel("Mean |SHAP| (embedding space)")
ax.set_title(f"Per-domino importance at transformer value head\n(decl_id={DECL_ID}=sixes trump, n={len(test_hands)} test states)")

legend_patches = [
    mpatches.Patch(color="#E91E63", label="Double"),
    mpatches.Patch(color="#FF9800", label="Count domino (5 or 10 pts)"),
    mpatches.Patch(color="#607D8B", label="Other"),
]
ax.legend(handles=legend_patches, fontsize=8)
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/14d_shap_per_domino.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved 14d_shap_per_domino.png")

Saved 14d_shap_per_domino.png


## 7. Comparison: transformer SHAP vs surrogate SHAP (14a)

14a found that `n_doubles` (mean |SHAP| = 4.84) and `trump_count` (4.39) dominated at the hand-feature level.  
Here we check whether the transformer similarly elevates doubles and count/trump dominoes.

In [11]:
# Compute grouped means: doubles, count dominoes, others
doubles_shap    = per_domino[per_domino["is_double"]]["mean_abs_shap"].mean()
count_shap      = per_domino[(~per_domino["is_double"]) & (per_domino["count_pts"] > 0)]["mean_abs_shap"].mean()
other_shap      = per_domino[(~per_domino["is_double"]) & (per_domino["count_pts"] == 0)]["mean_abs_shap"].mean()

print(f"Doubles (7 dominos):         mean |SHAP| = {doubles_shap:.5f}")
print(f"Count non-doubles (4 doms):  mean |SHAP| = {count_shap:.5f}")
print(f"Other (17 dominos):          mean |SHAP| = {other_shap:.5f}")

doubles_ratio = doubles_shap / other_shap
count_ratio   = count_shap / other_shap
print(f"\nDoubles elevation vs other: {doubles_ratio:.2f}x")
print(f"Count elevation vs other:   {count_ratio:.2f}x")

# Per-player breakdown
print("\nPer-player mean |SHAP| (all tokens):")
player_shap = rec_df.groupby("player")["token_shap"].mean()
labels_map = {0: "P0 (current)", 1: "P1 (opp)", 2: "P2 (partner)", 3: "P3 (opp)"}
for p, val in player_shap.items():
    print(f"  {labels_map[p]}: {val:.5f}")

Doubles (7 dominos):         mean |SHAP| = 0.16890
Count non-doubles (4 doms):  mean |SHAP| = 0.18820
Other (17 dominos):          mean |SHAP| = 0.14952

Doubles elevation vs other: 1.13x
Count elevation vs other:   1.26x

Per-player mean |SHAP| (all tokens):
  P0 (current): 0.15470
  P1 (opp): 0.15145
  P2 (partner): 0.15524
  P3 (opp): 0.17816


In [12]:
# Side-by-side comparison: transformer token SHAP (by category) vs 14a hand-feature SHAP
shap_14a = pd.read_csv(SHAP_14A_CSV)
print("14a features loaded:", shap_14a["feature"].tolist())

# Map categories to comparable summary statistics:
# Transformer side: doubles elevation, count elevation  
# 14a side: n_doubles, trump_count (proxies)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: 14a surrogate SHAP on hand features
ax = axes[0]
feat_colors = [
    "#E91E63" if f in ("n_doubles", "has_trump_double") else
    "#FF9800" if f in ("trump_count", "count_points") else
    "#607D8B"
    for f in shap_14a["feature"]
]
ax.barh(shap_14a["feature"][::-1], shap_14a["mean_abs_shap"][::-1], color=feat_colors[::-1])
ax.set_xlabel("Mean |SHAP| (points of game value)")
ax.set_title("14a: Surrogate GBM on\n10 hand features")
ax.set_xlim(0, shap_14a["mean_abs_shap"].max() * 1.15)

# Right: transformer per-domino SHAP top-15
ax = axes[1]
top15 = per_domino.head(15).copy()
tc_15 = [
    "#E91E63" if r.is_double else
    ("#FF9800" if r.count_pts > 0 else "#607D8B")
    for _, r in top15.iterrows()
]
ax.barh(top15["label"][::-1], top15["mean_abs_shap"][::-1], color=tc_15[::-1])
ax.set_xlabel("Mean |SHAP| (embedding space units)")
ax.set_title("14d: Transformer value head\nper-domino (top 15)")
ax.set_xlim(0, top15["mean_abs_shap"].max() * 1.15)

# Shared legend
legend_patches = [
    mpatches.Patch(color="#E91E63", label="Double / n_doubles"),
    mpatches.Patch(color="#FF9800", label="Count/trump related"),
    mpatches.Patch(color="#607D8B", label="Other"),
]
fig.legend(handles=legend_patches, loc="lower center", ncol=3, fontsize=9, frameon=True,
           bbox_to_anchor=(0.5, -0.05))
fig.suptitle("SHAP comparison: engineered-feature surrogate vs. transformer\n"
             "Both agree: doubles and count/trump dominos are most important",
             fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/14d_shap_comparison.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved 14d_shap_comparison.png")

14a features loaded: ['n_doubles', 'trump_count', 'n_singletons', 'count_points', 'total_pips', 'n_6_high', 'has_trump_double', 'max_suit_length', 'n_voids', 'n_5_high']


Saved 14d_shap_comparison.png


## 8. Waterfall-style examples: highest-V and lowest-V states

In [13]:
def token_level_shap_for_state(state_idx_in_test):
    """Return per-token |SHAP| and signed SHAP for a single test state."""
    sv = shap_vals[state_idx_in_test]  # (32, 357)
    signed  = sv.sum(axis=-1)           # (32,) — signed importance per token
    abs_imp = np.abs(sv).sum(axis=-1)   # (32,) — |SHAP| per token
    return signed, abs_imp


def plot_waterfall_state(state_idx_in_test, ax, title, top_n=12):
    """Bar chart of signed SHAP contributions for top tokens of one state."""
    hands = all_hands[N_BG + state_idx_in_test]
    signed, abs_imp = token_level_shap_for_state(state_idx_in_test)
    pred_val = test_values[state_idx_in_test]
    bg_mean  = wrapper(bg_embeds).detach().numpy().mean()

    # Build label for each non-context token
    labels = []
    for ti in range(1, 29):  # tokens 1-28
        flat = ti - 1
        p = flat // 7
        li = flat % 7
        gid = hands[p][li]
        pip_str = f"{DOMINO_HIGH[gid]}-{DOMINO_LOW[gid]}"
        player_str = ["P0", "P1", "P2", "P3"][p]
        labels.append((ti, f"{player_str}:{pip_str}", signed[ti], abs_imp[ti]))

    labels.sort(key=lambda x: -x[3])  # sort by |SHAP|
    top_labels = labels[:top_n]

    names  = [t[1] for t in top_labels]
    values = [t[2] for t in top_labels]
    clrs   = ["#2196F3" if v >= 0 else "#F44336" for v in values]

    ax.barh(range(len(names)), values, color=clrs)
    ax.set_yticks(range(len(names)))
    ax.set_yticklabels(names, fontsize=8)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_xlabel("Signed SHAP contribution")
    ax.set_title(f"{title}\npred={pred_val:.3f}, baseline={bg_mean:.3f}")


# Pick highest-V and lowest-V test state
best_idx  = int(np.argmax(test_values))
worst_idx = int(np.argmin(test_values))

print(f"Best  state (idx {best_idx}):  V = {test_values[best_idx]:.3f}")
print(f"Worst state (idx {worst_idx}): V = {test_values[worst_idx]:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
plot_waterfall_state(best_idx,  axes[0], "Highest-V state (top tokens)")
plot_waterfall_state(worst_idx, axes[1], "Lowest-V state (top tokens)")
plt.suptitle("Waterfall-style token SHAP: best vs worst predicted states", fontsize=12)
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/14d_shap_waterfall.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved 14d_shap_waterfall.png")

Best  state (idx 8):  V = 0.984
Worst state (idx 19): V = -0.990


Saved 14d_shap_waterfall.png


## 9. Summary statistics and findings

In [14]:
print("=" * 60)
print("SUMMARY: SHAP on Q-Val Transformer (14d)")
print("=" * 60)

print("\nTop 5 dominoes by mean |SHAP| at value head:")
top5 = per_domino.head(5)
for _, row in top5.iterrows():
    flag = "[DOUBLE]" if row.is_double else (f"[{int(row.count_pts)}pts]" if row.count_pts > 0 else "")
    print(f"  {row.label:6s}  {row.mean_abs_shap:.5f}  {flag}")

print(f"\nCategory elevation ratios (vs. non-count non-double baseline):")
print(f"  Doubles:           {doubles_ratio:.2f}x")
print(f"  Count non-doubles: {count_ratio:.2f}x")

print(f"\nPer-player mean |SHAP|:")
for p, val in player_shap.items():
    print(f"  {labels_map[p]}: {val:.5f}")

print("\n14a surrogate top-2: n_doubles (4.84), trump_count (4.39)")
print("14d transformer:     see top5 above — doubles and count dominos rank highest")

print("\nSHAP variant: GradientExplainer on concatenated embedding outputs (pre-input_proj)")
print(f"N_states={N_STATES}, N_bg={N_BG}, N_test={N_STATES-N_BG}, decl_id={DECL_ID} (sixes trump)")

SUMMARY: SHAP on Q-Val Transformer (14d)

Top 5 dominoes by mean |SHAP| at value head:
  6-3     0.33905  
  3-3     0.31347  [DOUBLE]
  5-5     0.24440  [DOUBLE]
  5-3     0.23017  
  6-4     0.20694  [10pts]

Category elevation ratios (vs. non-count non-double baseline):
  Doubles:           1.13x
  Count non-doubles: 1.26x

Per-player mean |SHAP|:
  P0 (current): 0.15470
  P1 (opp): 0.15145
  P2 (partner): 0.15524
  P3 (opp): 0.17816

14a surrogate top-2: n_doubles (4.84), trump_count (4.39)
14d transformer:     see top5 above — doubles and count dominos rank highest

SHAP variant: GradientExplainer on concatenated embedding outputs (pre-input_proj)
N_states=200, N_bg=100, N_test=100, decl_id=3 (sixes trump)


In [15]:
print("All outputs saved:")
print(f"  Figures: {FIGURES_DIR}/14d_shap_*.png")
print(f"  Tables:  {TABLES_DIR}/14d_shap_token_importance.csv")
print(f"           {TABLES_DIR}/14d_shap_per_domino.csv")

All outputs saved:
  Figures: /Users/jason/code/mk5-main/forge/analysis/results/figures/14d_shap_*.png
  Tables:  /Users/jason/code/mk5-main/forge/analysis/results/tables/14d_shap_token_importance.csv
           /Users/jason/code/mk5-main/forge/analysis/results/tables/14d_shap_per_domino.csv
